<a href="https://colab.research.google.com/github/bahmedx/RL_CartPole/blob/main/cartpole_hybrid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CartPole-v1: Hybrid Double-DQN Implementation

This notebook implements a **Hybrid Double Deep Q-Network (DDQN)** for the `CartPole-v1` environment.
It combines the **Double DQN architecture and gradient norm clipping** (from Model 1) with the **Optuna-tuned hyperparameters** (from Model 2) to eliminate overestimation bias while ensuring rapid, stable convergence.

In [ ]:
# Install dependencies
!pip install gymnasium[classic-control] torch numpy matplotlib imageio imageio-ffmpeg

In [ ]:
import os
import json
import random
import logging
from collections import deque
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import imageio
from IPython.display import Video, display

# Global Configuration
CONFIG = {
    "env_name": "CartPole-v1",
    "learning_rate": 0.0002704701663505405,  # Model 2 optimal LR
    "lr_decay": 0.9995483882722038,          # Gentle exponential LR decay
    "gamma": 0.994599791562965,              # Discount factor for long-term horizon
    "tau": 0.009433341411789989,             # Target soft-update rate
    "batch_size": 32,                        # Batch size
    "buffer_capacity": 20000,                # Experience replay buffer capacity
    "hidden_dim": 128,                       # Hidden layer dimensionality
    "epsilon_start": 1.0,
    "epsilon_decay": 0.9850294826912942,     # Exploration decay rate
    "epsilon_min": 0.01,
    "max_episodes": 400,
    "solved_score": 475.0,                   # Gym CartPole-v1 moving average target
    "seed": 42
}

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(CONFIG["seed"])
print("Environment and configuration initialized successfully.")

In [ ]:
# Neural Network and Replay Buffer Architecture
class DynamicQNetwork(nn.Module):
    def __init__(self, state_size: int, action_size: int, hidden_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_size, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_size)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

class ReplayBuffer:
    def __init__(self, capacity: int):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size: int):
        states, actions, rewards, next_states, dones = zip(*random.sample(self.buffer, batch_size))
        return np.array(states), np.array(actions), np.array(rewards), np.array(next_states), np.array(dones)

    def __len__(self):
        return len(self.buffer)

print("Network and Buffer classes defined.")

In [ ]:
# Hybrid Double-DQN Agent
class HybridDoubleDQNAgent:
    def __init__(self, state_size: int, action_size: int, params: dict):
        self.state_size = state_size
        self.action_size = action_size
        self.params = params
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # Networks
        self.q_network = DynamicQNetwork(state_size, action_size, params["hidden_dim"]).to(self.device)
        self.target_network = DynamicQNetwork(state_size, action_size, params["hidden_dim"]).to(self.device)
        self.target_network.load_state_dict(self.q_network.state_dict())

        # Optimizer and LR Scheduler
        self.optimizer = optim.Adam(self.q_network.parameters(), lr=params["learning_rate"])
        self.scheduler = optim.lr_scheduler.ExponentialLR(self.optimizer, gamma=params["lr_decay"])

        self.memory = ReplayBuffer(params["buffer_capacity"])
        self.epsilon = params["epsilon_start"]
        self.optimizer_stepped = False

    def act(self, state: np.ndarray) -> int:
        if np.random.rand() <= self.epsilon:
            return random.randrange(self.action_size)
        state_tensor = torch.FloatTensor(state).unsqueeze(0).to(self.device)
        with torch.no_grad():
            q_values = self.q_network(state_tensor)
        return int(np.argmax(q_values.cpu().data.numpy()))

    def step(self, state, action, reward, next_state, done):
        self.memory.push(state, action, reward, next_state, done)
        if len(self.memory) > self.params["batch_size"]:
            self.learn()

    def learn(self):
        states, actions, rewards, next_states, dones = self.memory.sample(self.params["batch_size"])

        states = torch.FloatTensor(states).to(self.device)
        actions = torch.LongTensor(actions).unsqueeze(1).to(self.device)
        rewards = torch.FloatTensor(rewards).unsqueeze(1).to(self.device)
        next_states = torch.FloatTensor(next_states).to(self.device)
        dones = torch.FloatTensor(dones).unsqueeze(1).to(self.device)

        # Double DQN decoupling: Action selected by main net, evaluated by target net
        with torch.no_grad():
            next_actions = self.q_network(next_states).argmax(1, keepdim=True)
            next_q = self.target_network(next_states).gather(1, next_actions)
            target_q = rewards + (self.params["gamma"] * next_q * (1 - dones))

        current_q = self.q_network(states).gather(1, actions)
        loss = nn.MSELoss()(current_q, target_q)

        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.q_network.parameters(), max_norm=1.0)
        self.optimizer.step()
        self.optimizer_stepped = True

        # Soft target update
        tau = self.params["tau"]
        for target_param, param in zip(self.target_network.parameters(), self.q_network.parameters()):
            target_param.data.copy_(tau * param.data + (1.0 - tau) * target_param.data)

    def update_epsilon_and_lr(self):
        self.epsilon = max(self.params["epsilon_min"], self.epsilon * self.params["epsilon_decay"])
        if self.optimizer_stepped:
            self.scheduler.step()

print("HybridDoubleDQNAgent loaded.")

In [ ]:
# Final Training Loop
env = gym.make(CONFIG["env_name"])
state_size = env.observation_space.shape[0]
action_size = env.action_space.n

agent = HybridDoubleDQNAgent(state_size, action_size, CONFIG)
scores = []
best_moving_avg = 0.0

print("Starting Hybrid Double-DQN Training...")

for episode in range(1, CONFIG["max_episodes"] + 1):
    state, _ = env.reset()
    score, done = 0.0, False

    while not done:
        action = agent.act(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        # Bootstrap on true termination only
        agent.step(state, action, reward, next_state, terminated)
        state = next_state
        score += reward

    agent.update_epsilon_and_lr()
    scores.append(score)

    moving_avg = np.mean(scores[-100:]) if len(scores) >= 100 else np.mean(scores)

    if episode % 25 == 0:
        print(f"Episode: {episode}/{CONFIG['max_episodes']} | Score: {score:.0f} | 100-Ep Avg: {moving_avg:.2f}")

    if moving_avg > best_moving_avg:
        best_moving_avg = moving_avg
        torch.save(agent.q_network.state_dict(), "hybrid_cartpole_dqn.pth")

    if moving_avg >= CONFIG["solved_score"] and episode >= 100:
        print(f"\nEnvironment solved early in {episode} episodes! 100-Episode Moving Average = {moving_avg:.2f}")
        break

env.close()

In [ ]:
# Training Performance Visualization
plt.figure(figsize=(10, 5))
plt.plot(scores, label="Raw Score", alpha=0.3, color="blue")
if len(scores) >= 100:
    moving_avg_line = np.convolve(scores, np.ones(100)/100, mode="valid")
    padded_avg = np.concatenate((np.full(99, np.nan), moving_avg_line))
    plt.plot(padded_avg, label="100-Episode Moving Avg", color="red", linewidth=2)
plt.axhline(CONFIG["solved_score"], color="green", linestyle="--", alpha=0.7, label="Solved Threshold (475)")
plt.title("Hybrid Double-DQN CartPole-v1 Performance")
plt.ylabel("Score (Steps Balanced)")
plt.xlabel("Episode")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.savefig("hybrid_training_curve.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Video Evaluation Simulation
print("Generating Evaluation Video...")
eval_env = gym.make(CONFIG["env_name"], render_mode="rgb_array")
q_net = DynamicQNetwork(state_size, action_size, CONFIG["hidden_dim"])
q_net.load_state_dict(torch.load("hybrid_cartpole_dqn.pth", map_location=agent.device))
q_net.eval()

state, _ = eval_env.reset(seed=CONFIG["seed"])
done = False
frames = []
steps = 0

while not done:
    frame = eval_env.render()
    frames.append(np.pad(frame, ((0, 0), (4, 4), (0, 0)), mode="edge"))
    state_tensor = torch.FloatTensor(state).unsqueeze(0).to(agent.device)
    with torch.no_grad():
        action = int(torch.argmax(q_net(state_tensor)).item())
    state, _, terminated, truncated, _ = eval_env.step(action)
    done = terminated or truncated
    steps += 1

eval_env.close()
print(f"Evaluation complete! Balanced for {steps} steps.")

video_path = "hybrid_cartpole_demo.mp4"
imageio.mimsave(video_path, frames, fps=30)
display(Video(video_path, embed=True))